In [ ]:
# Newer SDK: recovers if the model asks for a tool that doesn't exist
%pip install -q --no-cache-dir openai-agents==0.24.0 python-dotenv requests

In [ ]:
import os, requests
from openai import AsyncOpenAI
from IPython.display import display, Markdown
from google.colab import userdata                  # Colab's secret store
from agents import OpenAIChatCompletionsModel, set_tracing_disabled

groq_api_key   = userdata.get("GROQ_API_KEY")      # console.groq.com/keys
tavily_api_key = userdata.get("TAVILY_API_KEY")    # app.tavily.com

# Groq through the OpenAI-compatible address. max_retries: if the free limit is hit,
# wait and try again instead of failing straight away.
groq_client = AsyncOpenAI(api_key = groq_api_key,
                          base_url = "https://api.groq.com/openai/v1",
                          max_retries = 4)
set_tracing_disabled(True)                         # OpenAI traces need an OpenAI key

GROQ_MODEL       = "openai/gpt-oss-120b"           # best at using tools
GROQ_SMALL_MODEL = "openai/gpt-oss-20b"            # use if you hit the per-minute limit

print("Keys loaded:", groq_api_key[:5], tavily_api_key[:5])

In [ ]:
from agents import ModelSettings

def groq(model_name = GROQ_MODEL):
    """Wraps a Groq model name so the Agents SDK sends it to Groq exactly as written."""
    return OpenAIChatCompletionsModel(model = model_name, openai_client = groq_client)

# gpt-oss models think before answering. Keep that thinking on Groq's side: we only need the answer.
GROQ_SETTINGS = ModelSettings(extra_body = {"include_reasoning": False})

def print_markdown(text):
    display(Markdown(text))

def show_tool_calls(result):
    """Our replacement for OpenAI traces: which tools did the agent call, with what input, and what came back?"""
    calls = 0
    for item in result.new_items:
        if item.type == "tool_call_item":
            calls += 1
            raw = item.raw_item
            name = getattr(raw, "name", None) or (raw.get("name") if isinstance(raw, dict) else "?")
            args = getattr(raw, "arguments", None) or (raw.get("arguments") if isinstance(raw, dict) else "")
            print(f"🔧 Tool call {calls}: {name}({args})")
        elif item.type == "tool_call_output_item":
            print(f"   ↳ returned: {str(item.output)[:200]}...")
    if calls == 0:
        print("ℹ️ No tools were called: the answer came from the model's own knowledge.")
    u = result.context_wrapper.usage
    print(f"📊 Model requests: {u.requests} | input tokens: {u.input_tokens} | output tokens: {u.output_tokens}")

In [ ]:
TAVILY_URL = "https://api.tavily.com/search"   # Tavily API endpoint

def search_web(query: str, max_results: int = 3) -> str:
    """
    Search the web with Tavily and return a short summary of the top results.
    Use this for recent events, news, prices, product information or anything that may have changed.

    Args:
        query: What to search for, e.g. "Tesla Cybertruck price Canada".
        max_results: How many results to return (1 to 5, default 3).
    """
    max_results = max(1, min(int(max_results), 5))           # keep it small: saves free credits
    headers = {"Authorization": f"Bearer {tavily_api_key}",  # Tavily's documented way to send the key
               "Content-Type": "application/json"}
    payload = {"query": query, "max_results": max_results}

    try:
        response = requests.post(TAVILY_URL, json = payload, headers = headers, timeout = 30)
    except requests.RequestException as error:
        return f"Search failed: {error}"

    if response.status_code == 200:
        results = response.json().get("results", [])
        # Only the first 300 characters of each result: Groq's free plan allows about 8,000 tokens per minute
        summary = "\n".join(f"{i+1}. {r['title']}: {r['content'][:300]} (source: {r.get('url', '')})"
                            for i, r in enumerate(results))
        return summary if summary else "No relevant results found."
    return f"Tavily API error: {response.status_code}"

# ✅ Good habit: test the plain function BEFORE giving it to an agent
print(search_web("Tesla Cybertruck price Canada", 2))

In [ ]:
from agents import function_tool

# Turn the tested function into a tool. The docstring becomes the tool description the AI reads.
tavily_search = function_tool(search_web, name_override = "tavily_search")

print("Tool name:", tavily_search.name)
print("What the LLM sees:", tavily_search.description[:120], "...")
print("Parameters:", list(tavily_search.params_json_schema["properties"].keys()))

In [ ]:
from agents import Agent, Runner, SQLiteSession, RunConfig

session = SQLiteSession("live_researcher_practice")       # memory from Experiment 2

# If the model asks for a tool name that doesn't exist, tell it instead of crashing
run_config = RunConfig(tool_not_found_behavior = "return_error_to_model")

live_researcher_agent = Agent(name = "Live Market Researcher",
                              instructions = """
CONTEXT:
You are a world-class market research assistant with access to real-time web search via the tavily_search tool.

INSTRUCTION:
- Analyze the user's question and determine if recent or real-time information is needed.
- If the question involves recent events, news, prices or product info, always call tavily_search.
- Only use the tools you have been given. Do not call any other tool.
- Summarize search results clearly and concisely, do not copy-paste.
- Always start your answer with: "🔍 According to a web search …" when you used the tool.

INPUT:
You will receive a conversation history and the latest user question. Use the full context.

OUTPUT:
A clear, well-structured answer that integrates the search findings, with the source websites listed at the end.
""",
                              model = groq(),
                              model_settings = GROQ_SETTINGS,
                              tools = [tavily_search])          # <- NEW

print("✅ Agent created with Tavily tool.")

In [ ]:
q1 = "What are people saying about OpenAI's open-weight gpt-oss models?"
print_markdown(f"**User:** {q1}")

run1 = await Runner.run(starting_agent = live_researcher_agent, input = q1,
                        session = session, run_config = run_config)

print_markdown(f"### 🤖 Agent's Answer\n{run1.final_output}")

In [ ]:
show_tool_calls(run1)

In [ ]:
session = SQLiteSession("cybertruck_practice")

q1 = "What do reviewers say about the new Tesla Cybertruck?"
run1 = await Runner.run(live_researcher_agent, q1, session = session, run_config = run_config)
print_markdown(f"**Answer:** {run1.final_output}")

q2 = "Summarize the main features in one paragraph"
run2 = await Runner.run(live_researcher_agent, q2, session = session, run_config = run_config)
print_markdown(f"**Answer:** {run2.final_output}")
show_tool_calls(run2)

In [ ]:
@function_tool
def calculate_tariff_prices(base_price: float, start_tariff_pct: float, end_tariff_pct: float, step_pct: float) -> str:
    """
    Calculate the new price of a product for a range of import tariff rates.
    Use this whenever the user asks to simulate or calculate prices under different tariffs.

    Args:
        base_price: Price before tariff (a number, no currency symbol).
        start_tariff_pct: First tariff rate in percent, e.g. 5.
        end_tariff_pct: Last tariff rate in percent, e.g. 20.
        step_pct: Increase between rates in percent, e.g. 1.
    """
    if step_pct <= 0 or end_tariff_pct < start_tariff_pct:
        return "Invalid range: step must be > 0 and end must be >= start."
    rows = ["| Tariff % | New price | Increase |", "|---|---|---|"]
    rate = start_tariff_pct
    while rate <= end_tariff_pct + 1e-9:
        new_price = base_price * (1 + rate / 100)
        rows.append(f"| {rate:.1f}% | {new_price:,.2f} | {new_price - base_price:,.2f} |")
        rate += step_pct
    return "\n".join(rows)

print("✅ Tool ready:", calculate_tariff_prices.name)

In [ ]:
analyst_agent = Agent(name = "Analyst Agent",
                      instructions = """
CONTEXT:
You are a world-class market research assistant with two tools:
- tavily_search: real-time web search
- calculate_tariff_prices: exact price calculations for a range of tariff rates

INSTRUCTION:
- Use tavily_search for up-to-date facts, news, prices or product information.
- Use calculate_tariff_prices for any tariff or price simulation. Never do this maths in your head.
- If the task needs both, search first, then pass the price you found to the calculator.
- Only use these two tools. Do not call any other tool.
- Paraphrase search results; when using search, begin with "🔍 According to a web search …".
- Explain which price you used and why.

INPUT:
A conversation history and the latest user question.

OUTPUT:
A clear answer with the price found (with source), the full results table, and a 2-3 line summary.
""",
                      model = groq(),
                      model_settings = GROQ_SETTINGS,
                      tools = [tavily_search, calculate_tariff_prices])   # two custom tools

print("✅ Agent created with Tavily search + tariff calculator tools.")

In [ ]:
session = SQLiteSession("live_researcher")

q1 = ("Find the current price of a Tesla Cybertruck in Canada. "
      "Then, simulate how the price would change if import tariffs increased from 5% to 20% in 1% increments. "
      "Provide a summary of the results across all scenarios.")
print_markdown(f"**User:** {q1}")

run1 = await Runner.run(starting_agent = analyst_agent, input = q1, session = session, run_config = run_config)
print_markdown(f"### 🤖 Agent's Answer\n{run1.final_output}")
show_tool_calls(run1)